# Burrows-Wheeler Transform (BWT) and Pattern Matching

This notebook constructs the Burrows-Wheeler Transform of the *E. coli* genome and uses the BWT-based backward-search algorithm to count the occurrences of DNA reads in the genome.

**Input files:** `EcoliGenome.fa` and `readsetEcoliGenome.fa`.

## 1. Install and Import Libraries

Install the suffix-array library used to construct the BWT efficiently. The remaining operations use Biopython and NumPy.

In [ ]:
!pip -q install biopython pydivsufsort

import os
import numpy as np
from Bio import SeqIO
from pydivsufsort import divsufsort

## 2. Upload Input Files

Upload both FASTA files directly to the Colab working directory. The files will be accessible from `/content/`.

In [ ]:
from google.colab import files

uploaded = files.upload()

genome_path = '/content/EcoliGenome.fa'
reads_path = '/content/readsetEcoliGenome.fa'

assert os.path.exists(genome_path), 'EcoliGenome.fa was not uploaded.'
assert os.path.exists(reads_path), 'readsetEcoliGenome.fa was not uploaded.'

## 3. Read FASTA Files

Read all records from the genome and read-set FASTA files using `SeqIO.parse`. This supports files containing multiple FASTA records. Genome records are concatenated into one sequence, while each read is stored separately.

In [ ]:
genome_records = list(SeqIO.parse(genome_path, 'fasta'))
read_records = list(SeqIO.parse(reads_path, 'fasta'))

if not genome_records:
    raise ValueError('The genome FASTA file contains no sequences.')

if not read_records:
    raise ValueError('The read-set FASTA file contains no sequences.')

sequence = ''.join(str(record.seq).upper() for record in genome_records)
reads = [str(record.seq).upper() for record in read_records]

if '$' in sequence:
    raise ValueError("The genome sequence must not contain the '$' terminator.")

print('Genome length:', len(sequence))
print('Number of reads:', len(reads))
print('First read:', reads[0])

## 4. Construct the Burrows-Wheeler Transform

Append a unique terminal character (`$`) to the genome. Construct the suffix array and use it to obtain the BWT. This avoids explicitly generating and sorting every cyclic rotation, which would be impractical for a large genome.

In [ ]:
def BWT(sequence):
    text = sequence + '$'
    text_array = np.frombuffer(text.encode('ascii'), dtype=np.uint8)

    suffix_array = divsufsort(text_array)
    bwt_array = text_array[(suffix_array - 1) % len(text_array)]

    return bwt_array.tobytes().decode('ascii')


bwt_string = BWT(sequence)

print('BWT length:', len(bwt_string))
print('First 100 characters of BWT:')
print(bwt_string[:100])

## 5. Build the Occurrence and First-Column Information

Backward search requires the first-column cumulative counts and the number of occurrences of each character in the BWT prefix. To reduce memory usage, occurrence counts are stored at regular checkpoints and the remaining short interval is scanned when needed.

In [ ]:
def build_bwt_index(last_column, checkpoint=128):
    n = len(last_column)
    alphabet = sorted(set(last_column))
    char_to_index = {char: i for i, char in enumerate(alphabet)}

    counts = {char: last_column.count(char) for char in alphabet}

    first_occurrence = {}
    cumulative = 0
    for char in alphabet:
        first_occurrence[char] = cumulative
        cumulative += counts[char]

    number_of_blocks = (n + checkpoint - 1) // checkpoint
    occurrence_checkpoints = np.zeros(
        (len(alphabet), number_of_blocks + 1), dtype=np.int64
    )

    running_counts = np.zeros(len(alphabet), dtype=np.int64)

    for block in range(number_of_blocks):
        start = block * checkpoint
        end = min(start + checkpoint, n)
        block_counts = np.bincount(
            [char_to_index[c] for c in last_column[start:end]],
            minlength=len(alphabet)
        )
        running_counts += block_counts
        occurrence_checkpoints[:, block + 1] = running_counts

    return alphabet, char_to_index, first_occurrence, occurrence_checkpoints, checkpoint


alphabet, char_to_index, first_occurrence, occurrence_checkpoints, checkpoint = build_bwt_index(bwt_string)

print('Alphabet:', alphabet)
print('First occurrence:', first_occurrence)

## 6. Implement Backward Search (BWMatching)

The algorithm processes each pattern from right to left. At each step, it updates the interval of rows in the sorted first column that match the processed suffix. The final interval size gives the number of occurrences of the pattern.

In [ ]:
def rank(char, position):
    if char not in char_to_index:
        return 0

    block = position // checkpoint
    block_start = block * checkpoint
    char_index = char_to_index[char]

    count = occurrence_checkpoints[char_index, block]
    count += bwt_string[block_start:position].count(char)

    return int(count)


def pattern_count(pattern):
    top = 0
    bottom = len(bwt_string) - 1
    pattern = pattern.upper()

    while top <= bottom and pattern:
        symbol = pattern[-1]
        pattern = pattern[:-1]

        if symbol not in first_occurrence:
            return 0

        top = first_occurrence[symbol] + rank(symbol, top)
        bottom = first_occurrence[symbol] + rank(symbol, bottom + 1) - 1

    if top <= bottom:
        return bottom - top + 1

    return 0


def bwmatching(patterns):
    return [pattern_count(pattern) for pattern in patterns]


test_patterns = reads[:5]
print('Test patterns:', test_patterns)
print('Occurrence counts:', bwmatching(test_patterns))

## 7. Match All Reads

Apply BWMatching to every read in the input read set. The output contains the number of occurrences of each read in the genome.

In [ ]:
matching_counts = bwmatching(reads)

for i, (read, count) in enumerate(zip(reads, matching_counts)):
    print(f'Read {i + 1}: {read} | Occurrences: {count}')

total_reads = len(reads)
matched_reads = sum(count > 0 for count in matching_counts)

print('\nTotal reads:', total_reads)
print('Reads with at least one match:', matched_reads)
print('Reads with no match:', total_reads - matched_reads)

## 8. Summary

The genome was transformed using the Burrows-Wheeler Transform. The resulting BWT was indexed using cumulative character counts and occurrence checkpoints. Finally, backward search was used to count the occurrences of each read in the genome.